# 05 - Ensemble Ablation Study

**The core validation.** Does the three-prong ensemble beat any single prong?


In [ ]:
import numpy as np
from bgi_trident.graph.ensemble.stacker import ProngScores, TridentEnsemble
from bgi_trident.graph.ensemble.ablation import run_ablation, format_ablation_report

np.random.seed(42)
n = 1000
base = np.random.rand(n)
noise = lambda: np.random.normal(0, 0.15, n)
scores = ProngScores(
    pyg_scores=(base + noise()).clip(0,1).reshape(-1,1),
    dgl_scores=(base*0.8 + noise() + 0.1).clip(0,1).reshape(-1,1),
    xgb_scores=(base*0.7 + noise() + 0.15).clip(0,1).reshape(-1,1),
)
labels = (base > 0.5).astype(int)

results = run_ablation(scores, labels)
print(format_ablation_report(results))


## The Biryani-Every-Thursday Example

User 42 ordered from Meghana's every Thursday for 6 weeks, then stopped 3 weeks ago.


In [ ]:
ensemble = TridentEnsemble(calibrate=False)
ensemble.fit(scores, labels)

user42 = ProngScores(pyg_scores=np.array([[0.91]]), dgl_scores=np.array([[0.43]]), xgb_scores=np.array([[0.35]]))
breakdown = ensemble.predict_with_breakdown(user42)
print('User 42 - Meghana Biryani (stopped 3 weeks ago):')
for k, v in breakdown[0].items():
    print(f'  {k}: {v:.3f}')
print('\nPyG alone: 0.91 (wrong). Ensemble: ~0.38 (correct - user is drifting).')
